# 모델 평가와 Random Forest

## 철강산업 전력사용량 예측모형의 validation과 최종 평가

예측오차를 요약하는 MAE·RMSE·R²의 계산 방식과 해석 차이를 작은 숫자로 확인합니다. 이어서 Decision Tree의 불안정성과 Random Forest의 bootstrap·무작위 feature 선택·평균화 원리를 연결하고, fit·validation 성능, 선택모형의 최종 test와 큰 잔차가 발생한 조건을 비교합니다.


## 이번 교시의 학습목표

수업이 끝나면 다음을 수행할 수 있습니다.

- 개별 오차에서 MAE·MSE·RMSE를 계산하고, 큰 오차에 주는 가중과 단위의 차이를 설명합니다.
- R²를 test 실제값의 평균예측과 비교한 제곱오차의 상대적 수준으로 해석하고, 음수가 나올 수 있는 조건을 설명합니다.
- fit과 validation의 오차 격차로 과적합 가능성을 진단하고 Random Forest 깊이에 따른 성능 변화를 비교합니다.
- Random Forest의 bootstrap·무작위 feature 선택·tree 예측 평균화가 개별 tree의 불안정성을 완화하는 원리를 설명합니다.
- 같은 validation 행에서 평균 baseline·선형회귀·Decision Tree·Random Forest를 비교하고, 선택된 Pipeline 하나를 test에서 최종 평가한 뒤 절대오차가 큰 행의 실제값·예측값과 입력조건을 기록합니다.


## 이번 교시에서 사용할 데이터

### 강의 시연: UCI Steel Industry Energy Consumption

- **규모와 grain:** 35,040개의 15분 전력사용 관측
- **target:** `Usage_kWh`
- **비교모형:** 평균 baseline, Linear Regression, Decision Tree, Random Forest
- **오류를 해석할 조건:** 시간, 요일, 부하유형, power factor

5교시와 같은 자료를 사용합니다.

동일한 fit/validation 행을 네 후보에 사용하고, 선택을 마친 Pipeline 하나만 test에 적용하며, MAE·RMSE·R²뿐 아니라 가장 큰 잔차가 발생한 운전조건을 확인합니다.

### 적용 실습: UCI Bike Sharing

시간별 대여건수 `cnt`를 같은 네 모형으로 예측합니다. `cnt`의 범위는 1~977건이며 시간대와 근무일에 따라 분포가 크게 달라질 수 있습니다. 평균점수만 보고하지 않고 큰 오류가 발생한 날짜·시간·날씨 조건을 함께 조사합니다.
> `casual`과 `registered`는 누수 방지를 위해 계속 제외합니다.


## 모델 평가가 답해야 하는 질문

모형을 고를 때는 한 점수만 보는 대신 다음 근거를 함께 확인합니다.

1. 모든 후보가 같은 validation 행과 같은 target을 사용했는가
2. 평균 기준모형보다 사전에 정한 지표가 개선됐는가
3. fit과 validation의 오차 차이가 과도하지 않은가
4. 큰 오류가 어떤 관측과 조건에서 발생했는가
5. 선택된 Pipeline만 전체 train에 재적합해 test를 한 번 평가했는가

평균성능, 일반화 격차와 실패 관측을 연결해야 선택 근거가 재현됩니다.


## MAE·RMSE·R²가 강조하는 것

$$MAE=\frac{1}{n}\sum_i|y_i-\hat y_i|$$

$$RMSE=\sqrt{\frac{1}{n}\sum_i(y_i-\hat y_i)^2}$$

$$R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}$$

- MAE는 평균적인 오류 크기를 target 단위로 나타냅니다.
- RMSE는 제곱 때문에 드문 큰 오류를 더 강조하며 target 단위로 돌아옵니다.
- R²는 같은 평가자료의 평균예측과 제곱오차를 비교하는 무단위 상대지표입니다.

> `R²=0.8`은 평균적으로 20% 틀렸다는 뜻이 아닙니다. 물리적 오류 크기는 MAE와
RMSE로 함께 보고합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

actual = np.array([10.0, 20.0, 30.0, 40.0])
predictions = {
    "A: one large miss": np.array([9.0, 21.0, 29.0, 31.0]),
    "B: even misses": np.array([7.0, 17.0, 27.0, 37.0]),
}
row_tables, summary_rows = {}, []
for name, predicted in predictions.items():
    error = actual - predicted
    row_tables[name] = pd.DataFrame({
        "actual": actual,
        "predicted": predicted,
        "residual": error,
        "absolute_error": np.abs(error),
        "squared_error": error ** 2,
    })
    summary_rows.append({
        "model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": root_mean_squared_error(actual, predicted),
        "R2": r2_score(actual, predicted),
    })

print("[손계산으로 확인] 같은 MAE라도 큰 오류가 집중되면 RMSE가 커집니다.")
display(row_tables["A: one large miss"])
metric_table = pd.DataFrame(summary_rows)
display(metric_table.round(3))

fig, ax = plt.subplots(figsize=(7, 3.6))
x = np.arange(len(actual))
width = 0.36
for offset, (name, table) in zip([-width / 2, width / 2], row_tables.items()):
    ax.bar(x + offset, table["absolute_error"], width=width, label=name)
ax.set(xlabel="row", ylabel="absolute error", title="MAE can hide where large errors occur", xticks=x)
ax.legend()
plt.tight_layout()
plt.show()


## 과적합

| fit MAE | validation MAE | 검토할 가능성 |
|---:|---:|---|
| 1 | 2 | 두 표본에서 모두 낮은 오차 |
| 1 | 18 | fit의 세부 패턴에 과도하게 맞춘 가능성 |
| 20 | 22 | 격차는 작지만 두 표본 모두 예측력이 부족 |

fit–validation 격차만 작다고 충분한 모형은 아닙니다. validation 오차의 절대
크기와 평균 기준모형 대비 개선도 함께 확인합니다. 선택된 복잡도는 다른 seed나
기간에서도 다시 평가해야 합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

curve = pd.DataFrame({
    "tree_depth": [1, 2, 3, 4, 5, 6, 7, 8],
    "fit_MAE": [18, 14, 10, 7, 5, 3, 2, 1],
    "validation_MAE": [20, 15, 11, 9, 10, 13, 17, 22],
})
best_row = curve.loc[curve["validation_MAE"].idxmin()]

print("[곡선으로 확인] 복잡도가 커질수록 fit 오차는 줄어도 validation 오차는 다시 커질 수 있습니다.")
display(curve)
print(f"validation MAE가 가장 작은 깊이: {int(best_row.tree_depth)}")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(curve["tree_depth"], curve["fit_MAE"], marker="o", label="fit MAE")
ax.plot(curve["tree_depth"], curve["validation_MAE"], marker="o", label="validation MAE")
ax.axvline(best_row.tree_depth, color="#dc2626", linestyle="--", label="validation choice")
ax.set(xlabel="tree depth", ylabel="MAE", title="Model complexity and generalization gap")
ax.legend()
plt.tight_layout()
plt.show()


## Random Forest: 서로 다른 tree의 평균

Breiman (2001)의 Random Forest는 tree마다 서로 다른 bootstrap 학습표본을 만들고,
분기마다 일부 feature만 후보로 검토합니다. 회귀에서는 여러 tree의 예측을 평균합니다.

$$\hat y_{forest}=\frac{1}{B}\sum_{b=1}^{B}\hat y_b$$

한 행에 세 tree가 `90, 105, 105`를 예측하면 Forest 예측은 `100`입니다. 평균화는
단일 tree의 표본 민감성을 줄일 수 있지만, validation 성능이 자동으로 좋아진다는
보장은 없습니다.

출처: [Breiman (2001), *Random Forests*](https://doi.org/10.1023/A:1010933404324)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

rng = np.random.default_rng(42)
train_rows = np.arange(6)
bootstrap_samples = {
    f"tree_{tree_id}": rng.choice(train_rows, size=len(train_rows), replace=True)
    for tree_id in range(1, 4)
}
bootstrap_counts = pd.DataFrame({
    name: pd.Series(sample).value_counts().reindex(train_rows, fill_value=0)
    for name, sample in bootstrap_samples.items()
})
bootstrap_counts.index.name = "train_row"

tree_predictions = pd.Series([90.0, 105.0, 105.0], index=bootstrap_samples, name="prediction")
forest_prediction = tree_predictions.mean()

print("[표본과 평균으로 확인] bootstrap은 같은 행을 여러 번 뽑거나 제외할 수 있습니다.")
display(bootstrap_counts)
display(tree_predictions.to_frame())
print("Forest prediction = tree prediction mean =", forest_prediction)

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.bar(tree_predictions.index, tree_predictions.values, color="#60a5fa")
ax.axhline(forest_prediction, color="#dc2626", linestyle="--", label=f"mean = {forest_prediction:.1f}")
ax.set(ylabel="prediction", title="Random Forest averages different tree predictions")
ax.legend()
plt.tight_layout()
plt.show()


## 평균지표에서 실패 관측으로 내려가기

$$e_i=y_i-\hat y_i$$

| 확인 대상 | 목적 |
|---|---|
| 잔차 부호 | 과소예측과 과대예측 구분 |
| 절대오차 상위 행 | 큰 실패의 실제값·예측값 확인 |
| 시간·부하유형·power factor | 실패가 반복되는 조건 탐색 |
| 실제값–예측값 그림 | 대각선에서 크게 벗어난 범위 확인 |

큰 오류는 측정 오류, 드문 정상조건, feature 부족 등 여러 가설을 제시합니다. 원자료와
대조하기 전에는 원인으로 확정하거나 행을 삭제하지 않습니다.


## 평가 결과를 선택 근거로 작성하기

결과문에는 validation에서 사용한 주지표, 기준모형 대비 차이, fit–validation 격차,
선택 Pipeline의 최종 test 오차와 큰 오류 조건을 구분해 기록합니다.

Random Forest가 특정 알고리즘보다 보편적으로 우수하다고 결론내리지 않습니다.
선택한 표본·분할·지표 안에서 관찰된 우위와 아직 확인하지 않은 미래기간·외부시설
조건을 함께 제시합니다.


## 실행 환경과 입력자료

| 재현 조건 | 확인할 출력 |
|---|---|
| 입력자료 | 실제 파일 경로와 파일명 |
| 자료 구조 | 데이터 소개에 제시한 예상 `shape`와 필수 열 |
| 실행환경 | Python과 주요 라이브러리 버전 |

다음 셀은 분석에 사용한 파일과 실행환경을 기록합니다.

파일 경로 또는 `shape`가 강의 기준과 다르면 입력 파일과 버전을 확인한 뒤 분석을 진행합니다.


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행하십시오."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 한 행의 오차에서 MAE·RMSE 계산

세 행의 실제값과 예측값으로 오차, 절대오차와 제곱오차를 모두 만듭니다. 합계행을 이용해 MAE와 RMSE를 손으로 먼저 계산한 뒤 코드 결과와 비교합니다.


In [ ]:
metric_example = pd.DataFrame({
    "actual": [50.0, 70.0, 100.0],
    "predicted": [55.0, 60.0, 90.0],
})
metric_example["error"] = metric_example["actual"] - metric_example["predicted"]
metric_example["absolute_error"] = metric_example["error"].abs()
metric_example["squared_error"] = metric_example["error"] ** 2

mae_by_hand = metric_example["absolute_error"].sum() / len(metric_example)
mse_by_hand = metric_example["squared_error"].sum() / len(metric_example)
rmse_by_hand = np.sqrt(mse_by_hand)

print("[각 행의 오차가 지표의 재료가 되는 과정]")
display(metric_example)
print(f"절대오차 합 / 행 수 = {metric_example['absolute_error'].sum():.1f} / 3 = MAE {mae_by_hand:.2f}")
print(f"제곱오차 합 / 행 수 = {metric_example['squared_error'].sum():.1f} / 3 = MSE {mse_by_hand:.2f}")
print(f"sqrt(MSE) = RMSE {rmse_by_hand:.2f}")
print("MAE와 RMSE의 단위는 actual과 같은 단위입니다.")

r2_actual = np.array([10.0, 20.0, 30.0])
r2_prediction = np.array([15.0, 20.0, 25.0])
r2_mean = r2_actual.mean()
r2_sse = np.square(r2_actual - r2_prediction).sum()
r2_tss = np.square(r2_actual - r2_mean).sum()
r2_by_hand = 1 - r2_sse / r2_tss
print("\n[R²를 평균예측과 비교해 계산]")
display(pd.DataFrame({"actual": r2_actual, "prediction": r2_prediction, "mean_prediction": r2_mean}))
print(f"모형 SSE = {r2_sse:.1f}, 평균예측 TSS = {r2_tss:.1f}")
print(f"R² = 1 - {r2_sse:.1f}/{r2_tss:.1f} = {r2_by_hand:.2f}")
assert np.isclose(r2_by_hand, 0.75)


## 1. 평가 표본과 비교조건

후보 비교와 최종 평가의 자료 역할을 분리합니다.

| 단계 | 사용하는 행 | 목적 |
|---|---|---|
| fit | train 내부의 학습 행 | 전처리와 모형 적합 |
| validation | train 내부의 비교 행 | 주지표에 따른 후보·설정 선택 |
| test | 처음에 보관한 평가 행 | 선택된 Pipeline의 마지막 1회 평가 |

모든 후보는 같은 fit·validation 행과 같은 지표를 사용합니다. test에는 validation에서 정한 Pipeline 하나만 적용합니다. 분포표에서는 세 자료의 행 수, 평균, 표준편차와 범위를 확인합니다.



In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

data = pd.read_csv(find_course_data("steel/Steel_industry_data.csv"))
target_name = "Usage_kWh"
numeric_features = ["Lagging_Current_Reactive.Power_kVarh", "Leading_Current_Reactive_Power_kVarh", "Lagging_Current_Power_Factor", "Leading_Current_Power_Factor", "NSM"]
categorical_features = ["WeekStatus", "Day_of_week", "Load_Type"]
analysis_name = "Steel model validity"

feature_columns = numeric_features + categorical_features
X = data[feature_columns]
y = data[target_name]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_selection_fit, X_selection_validation, y_selection_fit, y_selection_validation = train_test_split(
    X_train, y_train, test_size=0.20, random_state=43
)
PRIMARY_METRIC = "MAE"  # 평균 오류를 중시하면 MAE, 큰 오류를 더 중시하면 RMSE
if PRIMARY_METRIC not in {"MAE", "RMSE"}:
    raise ValueError("PRIMARY_METRIC은 MAE 또는 RMSE여야 합니다.")
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])
print("fit/validation/test:", X_selection_fit.shape, X_selection_validation.shape, X_test.shape)


###  fit·validation·test의 target 범위

fit·validation·test 세 집단의 행 수와 사용량 분포가 얼마나 비슷하거나 다른지 확인합니다.


In [ ]:
split_summary = pd.DataFrame({
    "fit": y_selection_fit.describe(),
    "validation": y_selection_validation.describe(),
                    "test": y_test.describe(),
                }).round(2)
display(split_summary)


## 2. 세 지표와 fit–validation 격차

작은 예에서 지표의 차이를 계산한 뒤 같은 정의를 Steel의 모든 모델에 적용합니다.

| 지표 | 주요 질문 | 단위 |
|---|---|---|
| MAE | 한 관측에서 평균적으로 얼마나 틀렸는가 | kWh |
| RMSE | 큰 오류를 제곱 벌점으로 반영하면 오차가 얼마나 큰가 | kWh |
| R² | 각 split 실제값의 평균예측과 비교해 제곱오차가 얼마나 줄었는가 | 없음 |

실제 결과는 모델별 fit·validation 두 행, 모두 여덟 행으로 구성됩니다.

validation MAE와 `validation MAE − fit MAE`를 나란히 읽어 기준모형 대비 성능과 과적합 가능성을 구분합니다.


In [ ]:
models = {
    "mean baseline": DummyRegressor(strategy="mean"),
    "linear regression": Pipeline([
        ("prep", clone(preprocess)),
        ("model", LinearRegression()),
    ]),
    "decision tree": Pipeline([
        ("prep", clone(preprocess)),
        ("model", DecisionTreeRegressor(
            max_depth=None,
            min_samples_leaf=2,
            random_state=42,
        )),
    ]),
    "random forest": Pipeline([
        ("prep", clone(preprocess)),
        ("model", RandomForestRegressor(
            n_estimators=60,
            max_depth=14,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        )),
    ]),
}

rows = []
fitted_candidates = {}
validation_predictions = {}
for name, model in models.items():
    candidate = clone(model)
    candidate.fit(X_selection_fit, y_selection_fit)
    fitted_candidates[name] = candidate
    for split_name, split_X, split_y in [
        ("fit", X_selection_fit, y_selection_fit),
        ("validation", X_selection_validation, y_selection_validation),
    ]:
        prediction = candidate.predict(split_X)
        rows.append({
            "model": name,
            "split": split_name,
            "MAE": mean_absolute_error(split_y, prediction),
            "RMSE": root_mean_squared_error(split_y, prediction),
            "R2": r2_score(split_y, prediction),
        })
        if split_name == "validation":
            validation_predictions[name] = prediction

result_table = pd.DataFrame(rows)
selection_table = (
    result_table.loc[result_table["split"].eq("validation"), [
        "model", "MAE", "RMSE", "R2"
    ]]
    .sort_values(PRIMARY_METRIC)
    .reset_index(drop=True)
)
selection_rule = f"validation {PRIMARY_METRIC}가 가장 작은 후보"
metric_leader_name = selection_table.loc[0, "model"]
selected_model_name = metric_leader_name

comparison = result_table.pivot(
    index="model", columns="split", values=["MAE", "RMSE", "R2"]
)
comparison.columns = [f"{metric}_{split}" for metric, split in comparison.columns]
comparison[f"{PRIMARY_METRIC}_validation_minus_fit"] = (
    comparison[f"{PRIMARY_METRIC}_validation"]
    - comparison[f"{PRIMARY_METRIC}_fit"]
)

print(f"[fit과 validation에서 네 후보 비교 — 주지표: {PRIMARY_METRIC}]")
display(comparison.round(3).sort_values(f"{PRIMARY_METRIC}_validation"))
print("이 실습에서 사전에 정한 선택 규칙:", selection_rule)
print("선택 규칙에 따른 1차 후보:", selected_model_name)
print("fit–validation 격차와 큰 오류는 후보를 고른 뒤 함께 해석할 진단 근거입니다.")

# 후보와 설정을 정한 뒤 전체 train에 다시 적합하고 test를 한 번만 평가합니다.
final_model = clone(models[selected_model_name])
final_model.fit(X_train, y_train)
final_test_prediction = final_model.predict(X_test)
final_test_table = pd.DataFrame([{
    "selected_model": selected_model_name,
    "test_rows": len(y_test),
    "MAE": mean_absolute_error(y_test, final_test_prediction),
    "RMSE": root_mean_squared_error(y_test, final_test_prediction),
    "R2": r2_score(y_test, final_test_prediction),
}])


### 작은 오차 네 개를 계산한 뒤 실제 fit–validation 비교하기

먼저 MAE가 같아도 큰 오류가 있는 모델의 RMSE가 더 커지는 가상 예를 직접 계산합니다.

그다음 실제 네 후보의 fit과 validation MAE를 나란히 놓아 과적합 신호를 확인합니다.


In [ ]:
toy_actual = np.array([10.0, 20.0, 30.0, 40.0])
toy_predictions = {
    "A: 한 번 크게 틀림": np.array([11.0, 19.0, 29.0, 49.0]),
    "B: 고르게 틀림": np.array([13.0, 17.0, 33.0, 37.0]),
}
toy_error_view = pd.DataFrame({"actual": toy_actual})
toy_metric_rows = []
for toy_name, toy_prediction in toy_predictions.items():
    toy_error_view[f"{toy_name} prediction"] = toy_prediction
    toy_error_view[f"{toy_name} abs_error"] = np.abs(toy_actual - toy_prediction)
    toy_metric_rows.append({
        "model": toy_name,
        "MAE": mean_absolute_error(toy_actual, toy_prediction),
        "RMSE": root_mean_squared_error(toy_actual, toy_prediction),
        "R2": r2_score(toy_actual, toy_prediction),
    })
print("[1] 같은 네 건에서 직접 계산한 오차")
display(toy_error_view)
print("A와 B의 MAE는 같지만 큰 오류 9가 있는 A의 RMSE가 더 큽니다.")
display(pd.DataFrame(toy_metric_rows).round(3))

mae_pivot = result_table.pivot(index="model", columns="split", values="MAE")
mae_pivot["validation_minus_fit"] = mae_pivot["validation"] - mae_pivot["fit"]
print("[2] 실제 Steel 후보의 fit–validation MAE")
display(mae_pivot.sort_values("validation"))

import matplotlib.pyplot as plt
mae_pivot[["fit", "validation"]].plot(kind="bar", figsize=(8, 4), rot=0, title="Fit vs validation MAE")
plt.ylabel("MAE (kWh)")
plt.tight_layout()
plt.show()


## Random Forest 내부 확인: 개별 tree 예측의 평균

validation의 첫 행을 forest 내부의 모든 tree에 통과시킵니다.

개별 예측은 서로 다르지만 그 평균이 Random Forest의 최종 예측과 일치하는지 확인합니다.


In [ ]:
forest_pipeline = fitted_candidates["random forest"]
forest_preprocess = forest_pipeline.named_steps["prep"]
forest_model = forest_pipeline.named_steps["model"]

one_validation_row = X_selection_validation.iloc[[0]]
transformed_row = forest_preprocess.transform(one_validation_row)
individual_tree_predictions = np.array([
    tree.predict(transformed_row)[0]
    for tree in forest_model.estimators_
])
forest_prediction = float(forest_pipeline.predict(one_validation_row)[0])

tree_prediction_summary = pd.Series(individual_tree_predictions).describe().to_frame("kWh")
print("[첫 validation 행에 대한 앞 12개 tree의 예측]")
display(pd.DataFrame({
    "tree_number": np.arange(1, 13),
    "tree_prediction_kWh": individual_tree_predictions[:12],
}).round(3))
print(f"전체 {len(individual_tree_predictions)}개 tree 예측의 평균: {individual_tree_predictions.mean():.3f} kWh")
print(f"Random Forest Pipeline의 최종 예측       : {forest_prediction:.3f} kWh")
display(tree_prediction_summary.round(3))

assert np.isclose(individual_tree_predictions.mean(), forest_prediction)
print("Forest 평균화 확인: PASS")


## 3. 잔차와 실패 조건

$$residual_i=observed_i-predicted_i$$

평균지표에서 관측 수준의 실패로 분석 단위를 내려갑니다.

- 잔차가 양수이면 과소예측, 음수이면 과대예측
- 절대오차 상위 10개 행에서 실제값·예측값·입력조건 확인
- 실제값–예측값 그림에서 대각선과의 거리 확인
- 큰 오류가 부하유형·시간·power factor에 반복되는지 탐색

상위 오류행은 조사 대상입니다. 원자료 오류인지 드문 정상 운전조건인지 확인한 뒤 처리 방식을 결정합니다.

> **확인:** 큰 오류 다섯 건을 조사할 때 target과 예측값 외에 어떤 운전조건을 원자료에서 대조하겠습니까?


In [ ]:
metric_reference_name = selected_model_name
print(f"validation {PRIMARY_METRIC}로 정한 최종 평가 후보: {metric_reference_name}")
display(selection_table.round(3))
print("[선택을 끝낸 뒤 한 번 계산한 최종 test 결과]")
display(final_test_table.round(3))

error_table = X_test.copy()
error_table["observed"] = y_test
error_table["predicted"] = final_test_prediction
error_table["residual"] = error_table["observed"] - error_table["predicted"]
error_table["absolute_error"] = error_table["residual"].abs()
error_table["squared_error"] = error_table["residual"] ** 2
if "NSM" in error_table.columns:
    error_table["hour"] = (error_table["NSM"] // 3600).astype(int)
condition_columns = [column for column in [
    "hour", "hr", "workingday", "weathersit", "Load_Type",
    "WeekStatus", "Lagging_Current_Power_Factor", "temp", "hum",
] if column in error_table.columns]

large_error_summary = error_table.nlargest(10, "absolute_error")[[
    "observed", "predicted", "residual", "absolute_error", *condition_columns
]].copy()
display(large_error_summary)
print("\n[최종 평가 후보의 가장 큰 오류 5건]")
for row in error_table.nlargest(5, "absolute_error")[[
    "observed", "predicted", "absolute_error"
]].itertuples(index=False):
    print(
        f"- 실제 {row.observed:.0f}, 예측 {row.predicted:.1f}, "
        f"절대오차 {row.absolute_error:.1f}"
    )


### 실제값과 예측값

점이 대각선에서 멀수록 예측 오차가 큰 관측


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(error_table["observed"], error_table["predicted"], alpha=0.2, s=12)
limit = max(error_table["observed"].max(), error_table["predicted"].max())
ax.plot([0, limit], [0, limit], "--", color="black", linewidth=1)
ax.set(title=f"Observed vs predicted — {metric_reference_name}", xlabel="Observed kWh", ylabel="Predicted kWh")
plt.tight_layout()
plt.show()


## 4. 후보 선택·최종 평가 절차 검증

| 항목 | 확인 내용 |
|---|---|
| 평가행 | 네 후보가 같은 fit/validation index 사용하고 선택모형만 test에 적용 |
| 기준모형 | 평균 기준모형이 결과표에 포함 |
| 지표 | MAE·RMSE·R²가 모두 유한 |
| 정렬 | test 행 수와 예측 길이가 일치 |
| 일반화 격차 | 선택 후보의 validation 주지표 − fit 주지표 계산 |
| 오류 분석 | 절대오차 상위 행과 산점도 확인 |

이 검증은 선택한 holdout의 실행조건을 확인합니다. 다른 계절·공장·운영정책의 성능은 포함하지 않습니다.

In [ ]:
selected_fit_value = result_table.query(
    "model == @selected_model_name and split == 'fit'"
)[PRIMARY_METRIC].iloc[0]
selected_validation_value = result_table.query(
    "model == @selected_model_name and split == 'validation'"
)[PRIMARY_METRIC].iloc[0]
selected_generalization_gap = selected_validation_value - selected_fit_value

checks = pd.Series({
    "four models × fit and validation": result_table.shape[0] == 8,
    "finite selection metrics": np.isfinite(
        result_table[["MAE", "RMSE", "R2"]]
    ).all().all(),
    "selection uses validation only": set(selection_table["model"]) == set(models),
    "one selected model evaluated on test": len(final_test_table) == 1,
    "test prediction aligned": len(final_test_prediction) == len(y_test),
    "large-error cases available": len(error_table) == len(y_test),
    "selected fit-validation gap computed": np.isfinite(selected_generalization_gap),
}, name="passed")
display(checks.to_frame())
print(
    f"선택 후보 validation {PRIMARY_METRIC} - fit {PRIMARY_METRIC}:",
    selected_generalization_gap,
)
assert checks.all()
print("모델 평가조건 검증: PASS")


## ChatGPT 저장 응답 검증: 한 점수 대신 선택 근거 점검하기

평균 기준모형과 세 후보의 MAE·RMSE·R², fit–validation 격차와 큰 오류 조건을 함께 제공합니다. 요청의 목적은 ‘최고 모델’ 이름을 받는 것이 아니라 **어떤 근거와 우선순위로 비교할지** 확인하는 것입니다.

답변이 같은 평가행을 비교하는지, 큰 오류를 무시하지 않는지, 무작위 분할 결과를 미래성능으로 확대하지 않는지 점검합니다.


In [ ]:
review_prompt = f"""
다음 회귀모형의 후보 선택과 최종 평가를 검토해 주세요.

- 분석: {analysis_name}
- 사전에 정한 주지표: {PRIMARY_METRIC}
- validation 후보표:
{selection_table.to_string(index=False)}
- validation에서 정한 후보의 최종 test 결과:
{final_test_table.to_string(index=False)}
- 선택 후보의 validation {PRIMARY_METRIC} - fit {PRIMARY_METRIC}:
{selected_generalization_gap:.3f}
- 최종 test에서 절대오차가 큰 관측 10건:
{large_error_summary.to_string(index=False)}

아래 순서로 답해 주세요.
1. 기준모형 대비 validation 결과
2. fit–validation 격차가 시사하는 점
3. MAE와 RMSE가 다르게 강조하는 오류
4. 큰 절대오차가 반복되는 입력조건과 추가로 확인할 자료
5. 현재 평가로 확인하지 못한 미래기간 성능

각 판단에는 사용한 표의 열과 값을 적어 주세요. test 결과로 다른 모델이나
설정을 다시 고르지 말고, 실제 표에 없는 수치를 만들지 마세요.
""".strip()
print("[ChatGPT에 복사할 프롬프트]\n")
print(review_prompt)


### ChatGPT 저장 응답을 검증할 때

1. 추천 근거가 같은 평가행의 MAE·RMSE·R²를 사용하는지 확인합니다.
2. fit–validation 격차와 큰 오류 조건을 함께 언급했는지 확인합니다.
3. 운영 우선순위 없이 단순히 ‘가장 좋은 모델’이라고 쓴 답변은 수정합니다.


## 해석 범위

후보 비교는 train 내부의 동일한 validation 표본에서 계산한 MAE·RMSE·R²에 근거합니다. 선택된 Pipeline 하나만 무작위 test 표본에서 최종 평가하고 관측별 잔차를 확인했습니다.

계절 변화, 미래 기간의 분포 변화와 실제 의사결정 비용은 이 평가에 포함되지 않았습니다. 모델 선택 결과를 확장하려면 시간순 검증, 외부표본과 업무 비용을 추가로 검토해야 합니다.


## 오늘의 결론과 다음 단계

### 이번 교시의 결론

MAE·RMSE·R²의 질문을 구분하고, fit–validation 격차와 실패조건을 확인하면서 Random Forest의 평균화가 주는 변화를 비교합니다.

### 적용 순서

1. `06_model_evaluation_random_forest_follow.ipynb`에서 핵심 시연을 같은 조건으로 재현합니다.
2. `06_model_evaluation_random_forest_practice.ipynb`에서 지정된 데이터 또는 조건 하나만 바꿔 비교합니다.
3. 변경조건, 비교표 또는 그림, 결과문장과 현재 검증하지 못한 범위를 함께 기록합니다.

### 실습 중 확인할 질문

- 같은 예측오차에서 MAE와 RMSE는 무엇을 다르게 강조하며, R²는 어떤 기준선과 비교한 값입니까?
- 개별 Decision Tree의 학습자료가 조금만 바뀌어도 예측이 크게 변할 수 있는 이유는 무엇입니까?
- Random Forest는 bootstrap 표본과 무작위 feature로 만든 여러 tree의 회귀 예측을 어떻게 하나의 값으로 결합하며, 그 평균화가 왜 도움이 됩니까?

마지막 `PASS`는 Notebook에 적힌 실행조건을 확인한 결과이며 연구결론의 타당성을 대신하지 않습니다.
